# RDFLib Dataset / Named Graph 実践チュートリアル

教材：[Examples.datasets - RDFLib](https://rdflib.readthedocs.io/en/stable/apidocs/examples.datasets/?utm_source=chatgpt.com)

## Dataset の作成

In [1]:
from rdflib import Dataset, Graph, Literal, Namespace, URIRef

d: Dataset = Dataset()

ex: Namespace = Namespace("https://example.org/")
d.bind("ex", ex)

graph_1_id: URIRef = ex["graph-1"]
graph_2_id: URIRef = ex["graph-2"]

g1: Graph = d.graph(identifier=graph_1_id)

print("Dataset length:", len(d))
print("Graph 1 ID:", g1.identifier)
print("Default Graph ID:", d.default_graph.identifier)

Dataset length: 0
Graph 1 ID: https://example.org/graph-1
Default Graph ID: urn:x-rdflib:default


### Quad の追加

In [2]:
# Graph を直接操作して追加
g1.add(
    (
        ex["subject-x"],
        ex["predicate-x"],
        Literal("Triple X"),
    )
)

g1.add(
    (
        ex["subject-z"],
        ex["predicate-z"],
        Literal("Triple Z"),
    )
)

# Dataset に quad を直接追加
g2: Graph = d.graph(identifier=graph_2_id)
d.add(
    (
        ex["subject-y"],
        ex["predicate-y"],
        Literal("Triple Y"),
        g2,  # graph identifierも含めて指定
    )
)

print("Dataset length:", len(d))
print("Graph 1:", len(g1))
print("Graph 2:", len(g2))

Dataset length: 3
Graph 1: 2
Graph 2: 1


In [ ]:
# 中身チェック

print("=== Whole Dataset ===")
print(d.serialize(format="trig"))  # dataset を出力したいので TriG 形式

print("=== Graph 2 only ===")
print(g2.serialize(format="turtle"))

=== Whole Dataset ===
@prefix ex: <https://example.org/> .

ex:graph-2 {
    ex:subject-y ex:predicate-y "Triple Y" .
}

ex:graph-1 {
    ex:subject-x ex:predicate-x "Triple X" .

    ex:subject-z ex:predicate-z "Triple Z" .
}


=== Graph 2 only ===
@prefix ex: <https://example.org/> .

ex:subject-y ex:predicate-y "Triple Y" .




## Looping & Counting

### Dataset 全体の Quad を列挙する

In [4]:
quads = sorted(
    d.quads((None, None, None, None)),
    key=lambda q: (str(q[3]), str(q[0])),
)

for s, p, o, graph_id in quads:
    assert graph_id is not None
    print(
        graph_id.n3(d.namespace_manager),
        ":",
        s.n3(d.namespace_manager),
        p.n3(d.namespace_manager),
        o.n3(d.namespace_manager),
    )

ex:graph-1 : ex:subject-x ex:predicate-x "Triple X"
ex:graph-1 : ex:subject-z ex:predicate-z "Triple Z"
ex:graph-2 : ex:subject-y ex:predicate-y "Triple Y"


### Graph を限定して検索する

In [5]:
print("=== Quads in Graph 1 ===")

for s, p, o, graph_id in d.quads((None, None, None, graph_1_id)):
    print(s, p, o, graph_id)

print("=== Triples in Graph 1 ===")

for s, p, o in g1.triples((None, None, None)):
    print(s, p, o)

=== Quads in Graph 1 ===
https://example.org/subject-z https://example.org/predicate-z Triple Z https://example.org/graph-1
https://example.org/subject-x https://example.org/predicate-x Triple X https://example.org/graph-1
=== Triples in Graph 1 ===
https://example.org/subject-z https://example.org/predicate-z Triple Z
https://example.org/subject-x https://example.org/predicate-x Triple X


### default_union の違い

In [7]:
d.default_union = False

print("default_union=False")
print(list(d.triples((None, None, None))))

d.default_union = True

print("default_union=True")
print(list(d.triples((None, None, None))))

# 今後は元の設定に戻す
d.default_union = False


default_union=False
[]
default_union=True
[(rdflib.term.URIRef('https://example.org/subject-z'), rdflib.term.URIRef('https://example.org/predicate-z'), rdflib.term.Literal('Triple Z')), (rdflib.term.URIRef('https://example.org/subject-x'), rdflib.term.URIRef('https://example.org/predicate-x'), rdflib.term.Literal('Triple X')), (rdflib.term.URIRef('https://example.org/subject-y'), rdflib.term.URIRef('https://example.org/predicate-y'), rdflib.term.Literal('Triple Y'))]


### 同じ Triple を複数の Graph に追加

In [ ]:
shared = (
    ex["subject-z"],
    ex["predicate-z"],
    Literal("Triple Z"),
)

g2.add(shared)

print("Graph 1:", len(g1))
print("Graph 2:", len(g2))
print("Dataset length:", len(d))  # 重複して数えられることはない
print(
    "Quad count:",
    len(list(d.quads((None, None, None, None)))),
)

Graph 1: 2
Graph 2: 2
Dataset length: 3
Quad count: 4


## Manipulating Graphs

### Graph の列挙と Default Graph

In [10]:
print("=== Graph identifiers ===")

for graph in d.graphs():
    print(graph.identifier)

print()

# Default Graph に triple を追加する
d.add(
    (
        ex["subject-n"],
        ex["predicate-n"],
        Literal("Triple N"),
    )
)

print("Default Graph length:", len(d.default_graph))
print("Dataset length:", len(d))

print("=== Default Graph triples ===")
for triple in d.default_graph.triples((None, None, None)):
    print(triple)

=== Graph identifiers ===
https://example.org/graph-2
https://example.org/graph-1
urn:x-rdflib:default

Default Graph length: 1
Dataset length: 4
=== Default Graph triples ===
(rdflib.term.URIRef('https://example.org/subject-n'), rdflib.term.URIRef('https://example.org/predicate-n'), rdflib.term.Literal('Triple N'))


### Graph 自体を削除

In [13]:
# Graph 1 を削除
d.remove_graph(graph_1_id)

print("After removing Graph 1:")
for graph in d.graphs():
    print(graph.identifier, len(graph))
    
print()

# Default Graph の内容を削除
d.remove_graph(d.default_graph.identifier)

print("=== Remaining Dataset ===")
print(d.serialize(format="trig"))

After removing Graph 1:
https://example.org/graph-2 2
urn:x-rdflib:default 0

=== Remaining Dataset ===
@prefix ex: <https://example.org/> .

ex:graph-2 {
    ex:subject-y ex:predicate-y "Triple Y" .

    ex:subject-z ex:predicate-z "Triple Z" .
}




### 独立した Graph を Dataset に追加

In [14]:
# 明示的な Graph ID を持つ Graph
g_with_id: Graph = Graph(identifier=ex["graph-3"])

g_with_id.add(
    (
        ex["subject-k"],
        ex["predicate-k"],
        Literal("Triple K"),
    )
)

d.add_graph(g_with_id)

print(
    "Graph 3:",
    len(d.get_graph(ex["graph-3"])),
)

# ID を明示しない Graph
g_no_id: Graph = Graph()

g_no_id.add(
    (
        ex["subject-l"],
        ex["predicate-l"],
        Literal("Triple L"),
    )
)

d.add_graph(g_no_id)

print("Generated identifier:", g_no_id.identifier)
print("Identifier type:", type(g_no_id.identifier).__name__)

Graph 3: 1
Generated identifier: N0ef17eb65671481c86e1be1aa54569a8
Identifier type: BNode


## 補足

### 分離と共通

In [ ]:
world: Dataset = Dataset()
world.bind("ex", ex)

observed: Graph = world.graph(ex.observed)
inferred: Graph = world.graph(ex.inferred)
assumed: Graph = world.graph(ex.assumed)

# 観測事実
observed.add(
    (
        ex.wallet,
        ex.onTopOf,
        ex.table,
    )
)

# 推論された事実
inferred.add(
    (
        ex.wallet,
        ex.locatedIn,
        ex.kitchen,
    )
)

# 仮定
assumed.add(
    (
        ex.wallet,
        ex.locatedIn,
        ex.hallway,
    )
)

print("=== Wallet location by graph ===")

for s, p, o, graph_id in world.quads((ex.wallet, ex.locatedIn, None, None)):
    assert graph_id is not None
    print(
        graph_id.n3(world.namespace_manager),
        "→",
        o.n3(world.namespace_manager),
    )
    print(
        "  Triple:",
        s.n3(world.namespace_manager),
        p.n3(world.namespace_manager),
        o.n3(world.namespace_manager),
    )

=== Wallet location by graph ===
ex:inferred → ex:kitchen
  Triple: ex:wallet ex:locatedIn ex:kitchen
ex:assumed → ex:hallway
  Triple: ex:wallet ex:locatedIn ex:hallway


### 必要な Graph だけを統合

In [17]:
# 観測事実と推論事実だけを統合
trusted: Graph = observed + inferred

print("Trusted triples:", len(trusted))

print(
    "Kitchen:",
    (ex.wallet, ex.locatedIn, ex.kitchen) in trusted,
)

print(
    "Hallway:",
    (ex.wallet, ex.locatedIn, ex.hallway) in trusted,
)

Trusted triples: 2
Kitchen: True
Hallway: False


### Graph の出典を失わずにフィルタリング

In [18]:
selected_ids: set[URIRef] = {
    ex.observed,
    ex.inferred,
}

selected_quads = [
    quad for quad in world.quads((None, None, None, None)) if quad[3] in selected_ids
]

for s, p, o, graph_id in selected_quads:
    assert graph_id is not None
    print(
        graph_id.n3(world.namespace_manager),
        ":",
        s.n3(world.namespace_manager),
        p.n3(world.namespace_manager),
        o.n3(world.namespace_manager),
    )

ex:inferred : ex:wallet ex:locatedIn ex:kitchen
ex:observed : ex:wallet ex:onTopOf ex:table


### TriG による保存と復元

In [19]:
# Dataset → TriG
trig: str = world.serialize(format="trig")

print("=== Serialized TriG ===")
print(trig)

print()

# TriG → Dataset
restored: Dataset = Dataset()
restored.parse(data=trig, format="trig")

# Quad 単位で一致を確認
original_quads = set(world.quads((None, None, None, None)))

restored_quads = set(restored.quads((None, None, None, None)))

print("Restored correctly:", original_quads == restored_quads)

=== Serialized TriG ===
@prefix ex: <https://example.org/> .

ex:assumed {
    ex:wallet ex:locatedIn ex:hallway .
}

ex:inferred {
    ex:wallet ex:locatedIn ex:kitchen .
}

ex:observed {
    ex:wallet ex:onTopOf ex:table .
}



Restored correctly: True
